___
# Motor de KPI
___
**DESCRIPCIÓN :**
Este notebook ejecuta las configuraciones obtenidas del notebook `Configuracion`.


In [0]:
%run ../Funciones

In [0]:
# Diccionario que define los esquemas y las tablas que serán cargadas (<esquema> : [<tablas>])
dict_load_tables = {
    "markpi": ["Dim_Escenario"]    
}

# Carga las tablas definidas en el diccionario dentro de DataFrames para su posterior procesamiento
dfs = load_tables(dict_load_tables)

In [0]:
# Lista de identificadores de widgets
widgets = ['Escenario', 'Dia']

# Crear widgets del notebook de Databricks
create_widgets(widgets)

In [0]:
param = get_widget_values(widgets)

In [0]:
dias = int(param["Dia"]) if param["Dia"] else 0

hoy = datetime.today().date()
fecha_corte = hoy - timedelta(days=dias)

datekey_corte = int(fecha_corte.strftime("%Y%m%d"))

In [0]:
param.update(get_dimensions_description(dfs, param))
nombre_escenario = param.get("ScenarioName", "default").lower()
id_escenario = int(param["Escenario"])

In [0]:
config_path = f"{bronze_env}.configuracion.configuracion_{nombre_escenario}_Comentarios"
print(f" └ {config_path =}")

In [0]:
dfs["config"] = (
    read_table(config_path)
    .select(
        "id_kpi",
        "id_escenario",
        #"Id_Proceso",
        #"Id_Equipo",
        "Consulta_Diaria")
)

In [0]:
#display(dfs["config"])

In [0]:
df_config = dfs["config"]

In [0]:
dfs_resultados = []
consultas_fallidas = []

for row in df_config.toLocalIterator():
    try:
        df_tmp = spark.sql(row.Consulta_Diaria)

        df_tmp = (
            df_tmp
            .withColumn("id_kpi", F.lit(row.id_kpi))
            .withColumn("id_escenario", F.lit(row.id_escenario))
            .withColumn("Consulta_Diaria", F.lit(row.Consulta_Diaria))
        )

        dfs_resultados.append(df_tmp)

    except Exception as e:
        print(f"❌ Error en KPI {row.id_kpi}")
        print(f"   Consulta: {row.Consulta_Diaria[:120]}...")
        print(f"   Error: {str(e)}\n")

        consultas_fallidas.append({
            "id_kpi": row.id_kpi,
            "Consulta": row.Consulta_Diaria,
            "Error": str(e)
        })

In [0]:
if consultas_fallidas:
    print("\n==============================")
    print("CONSULTAS QUE FALLARON")
    print("==============================")

    for c in consultas_fallidas:
        print(f"KPI: {c['id_kpi']}")
        print(f"Error: {c['Error']}")
        print("----------------------")
else:
    print("Todas las consultas se ejecutaron correctamente")

In [0]:
if not dfs_resultados:
    raise Exception("❌ Ninguna consulta se ejecutó correctamente")
df_final = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs_resultados)

In [0]:
df_final = df_final.select(
    F.col("id_kpi").alias("Id_Kpi"),
    F.col("id_escenario").alias("Id_Escenario"),
    F.col("DateKey").alias("Id_Tiempo"),
    F.col("Diario").alias("Texto_Comentario")
)

In [0]:
df_final = df_final.withColumn(
    "Texto_Comentario",
    F.when(
        (F.col("Texto_Comentario").isNull()) |
        (F.trim(F.col("Texto_Comentario")) == "") |
        (F.col("Texto_Comentario") == "<NA>") |
        (F.col("Texto_Comentario") == "NaN"),
        None
    ).otherwise(F.col("Texto_Comentario"))
)

In [0]:
df_final = df_final.select(
    "Id_Kpi",
    "Id_Tiempo",
    "Texto_Comentario"
)
#display(df_final)

In [0]:
df_final = df_final.filter(
    F.col("Id_Tiempo") >= datekey_corte
)

In [0]:
df_final = df_final.withColumn(
    "Texto_Comentario",
    F.col("Texto_Comentario").cast("string")
)

In [0]:
#display(df_final)

In [0]:
table_name = f"{gold_env}.minajusta_operaciones.Fact_Kpi_Comentario"

if not spark.catalog.tableExists(table_name) or spark.table(table_name).limit(1).count() == 0:
    
    (
        df_final
        .write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(table_name)
    )

else:

    spark.sql(f"""
        DELETE FROM {table_name}
        WHERE Id_Tiempo >= {datekey_corte}
    """)

    (
        df_final
        .write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(table_name)
    )

In [0]:

df_gold = spark.table(table_name)

w_global = Window.orderBy("Id_Tiempo","Id_Kpi")

df_gold = df_gold.withColumn(
    "Id_Fact_Kpi_Comentario",
    F.row_number().over(w_global)
)

df_gold = df_gold.select(
    "Id_Fact_Kpi_Comentario",
    "Id_Kpi",
    #"IdEscenario",
    "Id_Tiempo",
    "Texto_Comentario"
)

(
    df_gold
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)